<a href="https://colab.research.google.com/github/hayappi-m/J_Quants_API/blob/main/J_Quants_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
import json
import requests
import gspread
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
from typing import List, Dict, Any
from google.colab import auth
from google.auth import default

# --- 認証 ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# --- 設定項目 ---
SOURCE_SPREADSHEET_ID = '10DFaSz9TMbF1cTVArNz701la_Eb2RVF1G08IoA80nhw'
JQUANTS_API_KEY = '_5-xfDWpyy4F9WRJxktdMKQhtNdJbGUXXMfMyTWmd-c'
OUTPUT_DIR = 'output_json'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- パラメーター (必要に応じてここを変更してください) ---
TARGET_SHEET_NAMES = ["資産運用", "資産運用シム"]  # 読み込み対象のシート名リスト
MA5_DAYS, MA5_AGO = 5, 3
MA25_DAYS, MA25_AGO = 25, 5
MA75_DAYS, MA75_AGO = 75, 10
SWING_WINDOW, SWING_LOOKBACK = 5, 100
HIGH_LOW_PERIOD, VOL_MA_PERIOD = 20, 20
BB_PERIOD, BB_STD_DEV = 20, 2.0
HISTORICAL_BARS_DAYS = 500
CUTOFF_TIME = "16:30"

# --- 関数定義 ---
def is_before_cutoff():
    jst = pytz.timezone('Asia/Tokyo')
    now = datetime.now(jst).time()
    cutoff = datetime.strptime(CUTOFF_TIME, "%H:%M").time()
    return now < cutoff

def fetch_listed_info_v2(api_key: str) -> Dict[str, Any]:
    url = "https://api.jquants.com/v2/equities/master"
    headers = {"x-api-key": api_key}
    try:
        res = requests.get(url, headers=headers, timeout=30)
        return {item["Code"]: item for item in res.json().get("data", [])} if res.status_code == 200 else {}
    except: return {}

def fetch_fins_summary_v2(code: str, api_key: str) -> Dict[str, Any]:
    headers = {"x-api-key": api_key}
    url = f"https://api.jquants.com/v2/fins/summary?code={code}"
    try:
        res = requests.get(url, headers=headers, timeout=15)
        if res.status_code == 200:
            data = res.json().get("data", [])
            return data[-1] if data else {}
        return {}
    except Exception as e:
        print(f"財務情報取得エラー ({code}): {e}")
        return {}

def fetch_earnings_date_v2(code: str, api_key: str) -> str:
    headers = {"x-api-key": api_key}
    url = f"https://api.jquants.com/v2/fins/earnings-date?code={code}"
    try:
        res = requests.get(url, headers=headers, timeout=15)
        if res.status_code == 200:
            data = res.json().get("data", [])
            if data:
                return data[-1].get("SchDate", "")
        return ""
    except Exception as e:
        print(f"決算発表予定日取得エラー ({code}): {e}")
        return ""

def fetch_all_historical_bars(code: str, api_key: str) -> List[Dict[str, Any]]:
    headers = {"x-api-key": api_key}
    url = f"https://api.jquants.com/v2/equities/bars/daily?code={code}"
    try:
        res = requests.get(url, headers=headers, timeout=30)
        return res.json().get("data", []) if res.status_code == 200 else []
    except: return []

def calculate_technical_indicators(bars: List[Dict[str, Any]]) -> pd.DataFrame:
    if not bars: return pd.DataFrame()
    df = pd.DataFrame(bars)
    for col in ['O', 'H', 'L', 'C', 'Vo']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 移動平均線および過去営業日前の移動平均線
    df['SMA5'] = df['C'].rolling(MA5_DAYS).mean()
    df['SMA5_DAYS'] = df['SMA5'].shift(MA5_AGO)

    df['SMA25'] = df['C'].rolling(MA25_DAYS).mean()
    df['SMA25_DAYS'] = df['SMA25'].shift(MA25_AGO)

    df['SMA75'] = df['C'].rolling(MA75_DAYS).mean()
    df['SMA75_DAYS'] = df['SMA75'].shift(MA75_AGO)

    # ボリンジャーバンド (+2σ)
    sma_bb = df['C'].rolling(BB_PERIOD).mean()
    std_bb = df['C'].rolling(BB_PERIOD).std(ddof=0)
    df['BB_2sigma'] = sma_bb + (BB_STD_DEV * std_bb)

    # 指定期間の高値・安値
    df['High_Period'] = df['H'].rolling(HIGH_LOW_PERIOD).max()
    df['Low_Period'] = df['L'].rolling(HIGH_LOW_PERIOD).min()

    # 出来高の移動平均
    df['Vol_SMA'] = df['Vo'].rolling(VOL_MA_PERIOD).mean()

    # スイング高値・安値の検出
    sh_list = []
    sl_list = []

    for i in range(len(df)):
        start_idx = max(0, i - SWING_LOOKBACK + 1)
        sub_df = df.iloc[start_idx : i + 1]

        if len(sub_df) >= (2 * SWING_WINDOW + 1):
            window_start = max(0, i - SWING_WINDOW)
            window_end = min(len(df) - 1, i + SWING_WINDOW)
            h_slice = df['H'].iloc[window_start : window_end + 1]
            l_slice = df['L'].iloc[window_start : window_end + 1]

            is_sh = (df['H'].iloc[i] == h_slice.max()) and (len(h_slice) == (2 * SWING_WINDOW + 1))
            is_sl = (df['L'].iloc[i] == l_slice.min()) and (len(l_slice) == (2 * SWING_WINDOW + 1))

            sh_list.append(df['H'].iloc[i] if is_sh else np.nan)
            sl_list.append(df['L'].iloc[i] if is_sl else np.nan)
        else:
            sh_list.append(np.nan)
            sl_list.append(np.nan)

    df['Temp_SH'] = sh_list
    df['Temp_SL'] = sl_list

    latest_sh_col, prev_sh_col = [], []
    latest_sl_col, prev_sl_col = [], []
    valid_shs = []
    valid_sls = []

    for i in range(len(df)):
        if not np.isnan(df['Temp_SH'].iloc[i]):
            valid_shs.append(df['Temp_SH'].iloc[i])
        if not np.isnan(df['Temp_SL'].iloc[i]):
            valid_sls.append(df['Temp_SL'].iloc[i])

        latest_sh_col.append(valid_shs[-1] if len(valid_shs) > 0 else np.nan)
        prev_sh_col.append(valid_shs[-2] if len(valid_shs) > 1 else np.nan)

        latest_sl_col.append(valid_sls[-1] if len(valid_sls) > 0 else np.nan)
        prev_sl_col.append(valid_sls[-2] if len(valid_sls) > 1 else np.nan)

    df['Latest_SH'] = latest_sh_col
    df['Prev_SH'] = prev_sh_col
    df['Latest_SL'] = latest_sl_col
    df['Prev_SL'] = prev_sl_col

    # 陽線・陰線判定
    is_bullish = df['C'] >= df['O']
    candlestick_type = ["陽" if b else "陰" for b in is_bullish]

    # U_Col: 25日線下での組み合わせ判定
    below_sma25 = df['C'] < df['SMA25']
    u_col_list = []
    for i in range(len(df)):
        if i > 0 and below_sma25.iloc[i] and below_sma25.iloc[i-1]:
            u_col_list.append(f"{candlestick_type[i-1]}{candlestick_type[i]}")
        else:
            u_col_list.append("")
    df['U_Col'] = u_col_list

    # V_Col: 25日移動平均線の上抜け判定
    golden_cross = (df['C'].shift(1) <= df['SMA25'].shift(1)) & (df['C'] > df['SMA25'])
    df['V_Col'] = [f"上({candlestick_type[i]})" if golden_cross.iloc[i] else "" for i in range(len(df))]

    # W_Col: 陽線の連続日数カウント
    w_col_list, streak = [], 0
    for b in is_bullish:
        streak = streak + 1 if b else 0
        w_col_list.append(streak)
    df['W_Col'] = w_col_list

    # ご指定いただいた項目の順番に整列
    desired_columns = [
        'Date', 'O', 'H', 'L', 'C', 'Vo', 'BB_2sigma',
        'SMA5', 'SMA5_DAYS', 'SMA25', 'SMA25_DAYS', 'SMA75', 'SMA75_DAYS',
        'Latest_SH', 'Prev_SH', 'High_Period', 'Low_Period',
        'Latest_SL', 'Prev_SL', 'Vol_SMA', 'U_Col', 'V_Col', 'W_Col'
    ]

    for col in desired_columns:
        if col not in df.columns:
            df[col] = np.nan

    return df[desired_columns]

# --- メイン処理 ---
def fetch_and_create_stock_data():
    source_ss = gc.open_by_key(SOURCE_SPREADSHEET_ID)
    target_items = []
    seen_codes = set()
    saved_count = 0

    for sheet_name in TARGET_SHEET_NAMES:
        try:
            ws = source_ss.worksheet(sheet_name)
            for row in ws.get_all_values()[4:]:
                if len(row) > 2 and row[2].strip():
                    code = str(row[2]).strip()
                    if code not in seen_codes:
                        seen_codes.add(code)
                        target_items.append({"code": code, "aj_val": row[35] if len(row) > 35 else ""})
        except gspread.exceptions.WorksheetNotFound:
            print(f"スキップ: シート '{sheet_name}' が見つかりませんでした。")
        except Exception as e:
            print(f"シート '{sheet_name}' 読み込みエラー: {e}")

    master_map = fetch_listed_info_v2(JQUANTS_API_KEY)
    cutoff_mode = is_before_cutoff()

    for item_info in target_items:
        code = item_info["code"]
        if len(code) != 5 or item_info["aj_val"] == "利確": continue

        print(f"処理中: {code}")

        fins_data = fetch_fins_summary_v2(code, JQUANTS_API_KEY)
        SchDate = fetch_earnings_date_v2(code, JQUANTS_API_KEY)
        fins_data["SchDate"] = SchDate

        raw_bars = fetch_all_historical_bars(code, JQUANTS_API_KEY)
        df_indicators = calculate_technical_indicators(raw_bars)

        target_df = df_indicators.iloc[:-1] if cutoff_mode and len(df_indicators) > 1 else df_indicators
        metadata = {**master_map.get(code, {}), **fins_data}

        data_to_save = {
            "metadata": metadata,
            "history": target_df.tail(HISTORICAL_BARS_DAYS).to_dict(orient="records")
        }

        file_path = os.path.join(OUTPUT_DIR, f"{code}.json")
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data_to_save, f, ensure_ascii=False, indent=4)

        saved_count += 1
        time.sleep(0.5)

    return saved_count

if __name__ == "__main__":
    start_time = time.time()
    total_saved = fetch_and_create_stock_data()
    elapsed = time.time() - start_time
    print(f"\n全処理完了: 生成ファイル数 {total_saved}件 / {int(elapsed // 60)}分 {int(elapsed % 60)}秒")

処理中: 26780
処理中: 28970
処理中: 45740
処理中: 45870
処理中: 47550
処理中: 61940
処理中: 93480
処理中: 98430
処理中: 24130
処理中: 44830
処理中: 30030
処理中: 46840
処理中: 90410
処理中: 77400
処理中: 95010
処理中: 16050
処理中: 98310
処理中: 25310
処理中: 42050
処理中: 50190
処理中: 28710
処理中: 43680
処理中: 43690
処理中: 55920
処理中: 77300
処理中: 23260
処理中: 40710
処理中: 23710
処理中: 77470
処理中: 47760
処理中: 43730
処理中: 44310
処理中: 277A0
処理中: 69460
処理中: 65240
処理中: 58890
処理中: 79360
処理中: 63230
処理中: 88020
処理中: 19680
処理中: 25850
処理中: 53840
処理中: 99620
処理中: 50160
処理中: 19410
処理中: 14070
処理中: 30760
処理中: 81360
処理中: 31040
処理中: 31100
処理中: 53930
処理中: 71640
処理中: 547A0
処理中: 58320
処理中: 73270
処理中: 71670
処理中: 71720
処理中: 71800
処理中: 71890
処理中: 82530
処理中: 83040
処理中: 83060
処理中: 83080
処理中: 83340
処理中: 84250
処理中: 83410
処理中: 83580
処理中: 86040
処理中: 85850
処理中: 58310
処理中: 71820
処理中: 83310
処理中: 85240
処理中: 86010
処理中: 71860
処理中: 73370
処理中: 83090
処理中: 83590
処理中: 84180
処理中: 23840
処理中: 90030
処理中: 90090
処理中: 84390
処理中: 90200
処理中: 90310
処理中: 90420
処理中: 90440
処理中: 90060
処理中: 90070
処理中: 90100
処理中: 93020

In [2]:
import os
import time
import json
import gspread
import pandas as pd
from gspread.exceptions import APIError

# ==========================================
# --- ★各種パラメーター設定 ---
# ==========================================
OUTPUT_DIR = 'output_json'
TARGET_SPREADSHEET_ID = '10DFaSz9TMbF1cTVArNz701la_Eb2RVF1G08IoA80nhw'
TARGET_SHEET_NAME = "All_Stocks_Summary"
LATEST_ROWS_COUNT = 20
CHUNK_SIZE = 1000

# ▼▼▼ 不要な列を「改行」や「スペース」で区切って文字として書くだけでOKです ▼▼▼
EXCLUDE_TEXT = """
    CoNameEn
    S17
    S33
    ScaleCat
    Mkt
    Mrgn
    MrgnNm
    ProdCat
    DiscNo
    DocType
    CurPerSt
    CurPerEn
    CurFYSt
    CurFYEn
    NxtFYSt
    NxtFYEn
    DEPS
    CFO
    CFI
    CFF
    CashEq
    Div1Q
    Div2Q
    Div3Q
    DivFY
    DivAnn
    DivUnit
    FDiv1Q
    FDiv2Q
    FDiv3Q
    FDivFY
    FDivUnit
    FDivTotalAnn
    FPayoutRatioAnn
    NxFDiv1Q
    NxFDiv2Q
    NxFDiv3Q
    NxFDivFY
    NxFDivAnn
    NxFDivUnit
    NxFPayoutRatioAnn
    FSales2Q
    FOP2Q
    FOdP2Q
    FNP2Q
    FEPS2Q
    NxFSales2Q
    NxFOP2Q
    NxFOdP2Q
    NxFNp2Q
    NxFEPS2Q
    NxFSales
    NxFOP
    NxFOdP
    NxFNp
    NxFEPS
    MatChgSub
    SigChgInC
    ChgByASRev
    ChgNoASRev
    ChgAcEst
    RetroRst
    NCSales
    NCOP
    NCOdP
    NCNP
    NCEPS
    NCTA
    NCEq
    NCEqAR
    NCBPS
    FNCSales2Q
    FNCOP2Q
    FNCOdP2Q
    FNCNP2Q
    FNCEPS2Q
    NxFNCSales2Q
    NxFNCOP2Q
    NxFNCOdP2Q
    NxFNCNP2Q
    NxFNCEPS2Q
    FNCSales
    FNCOP
    FNCOdP
    FNCNP
    FNCEPS
    NxFNCSales
    NxFNCOP
    NxFNCOdP
    NxFNCNP
    NxFNCEPS
    NCShEq
    NCROE
    AdjFactor
    AdjO
    AdjH
    AdjL
    AdjC
    AdjVo
    DivTotalAnn
    PayoutRatioAnn
    UL
    LL
    SMA_BB
    STD_BB
    ExRT
"""
# 自動でPythonがリストに変換します（クォーテーションもカンマも不要！）
EXCLUDE_COLUMNS = [col.strip() for col in EXCLUDE_TEXT.split() if col.strip()]


# ▼▼▼【重要】スプレッドシートの左側から順に並べたい列を指定してください ▼▼▼
# ※ こちらも改行やスペースで区切るだけでOKです（クォーテーションもカンマも不要！）
PREFERRED_TEXT = """
    Code
    MktNm
    CoName
    SchDate
    Date
    O
    H
    L
    C
    BB_2sigma
    SMA5
    SMA5_DAYS
    SMA25
    SMA25_DAYS
    SMA75
    SMA75_DAYS
    High_Period
    Low_Period
    Latest_SH
    Prev_SH
    Latest_SL
    Prev_SL
    Vo
    Vol_SMA
    U_Col
    V_Col
    W_Col
"""
# 自動でPythonがリストに変換します
PREFERRED_COLUMNS = [col.strip() for col in PREFERRED_TEXT.split() if col.strip()]
# ==========================================

# --- シート準備 ---
source_ss = gc.open_by_key(TARGET_SPREADSHEET_ID)
try:
    ws = source_ss.worksheet(TARGET_SHEET_NAME)
except:
    ws = source_ss.add_worksheet(title=TARGET_SHEET_NAME, rows="100000", cols="100")
ws.clear()

# --- メイン処理 ---
json_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')]
all_data_list = []

print(f"合計 {len(json_files)} 個のファイルを処理中（各銘柄 {LATEST_ROWS_COUNT} 行抽出）...")

success_count = 0
skip_count = 0

for file_name in json_files:
    file_path = os.path.join(OUTPUT_DIR, file_name)
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        history = data.get("history", [])
        meta = data.get("metadata", {})

        if not history:
            skip_count += 1
            continue

        df = pd.DataFrame(history).tail(LATEST_ROWS_COUNT).iloc[::-1].copy()

        # メタデータを結合（ただし 'Date' と除外リストにある列は追加しない）
        for key, value in meta.items():
            if key == 'Date' or key in EXCLUDE_COLUMNS:
                continue
            df[key] = value

        all_data_list.append(df)
        success_count += 1

    except Exception as e:
        print(f"読み込みエラー ({file_name}): {e}")
        skip_count += 1

print(f"処理完了: 成功 {success_count} ファイル, スキップ/エラー {skip_count} ファイル")

# --- 分割書き出し（チャンク処理 ＆ 行数自動拡張） ---
if all_data_list:
    final_df = pd.concat(all_data_list, ignore_index=True)

    # 念のため、結合後にDataFrameの列の中にEXCLUDE_COLUMNSが残っていればここで一括削除
    final_df = final_df.drop(columns=[col for col in EXCLUDE_COLUMNS if col in final_df.columns], errors='ignore')

    # ▼▼▼ 列の並び替え処理 ▼▼▼
    existing_preferred = [col for col in PREFERRED_COLUMNS if col in final_df.columns]
    other_columns = [col for col in final_df.columns if col not in existing_preferred]

    # 指定した順番の列 ＋ 残りの列 を結合して再配置
    final_df = final_df[existing_preferred + other_columns]
    # ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

    final_df = final_df.fillna("")

    values = [final_df.columns.tolist()] + final_df.values.tolist()
    total_rows = len(values)

    current_row_count = ws.row_count
    if total_rows > current_row_count:
        print(f"シートの行数が不足しているため拡張します: 現在 {current_row_count} 行 → 必要 {total_rows + 1000} 行")
        ws.resize(rows=total_rows + 1000)

    print(f"全 {total_rows} 行のデータを {CHUNK_SIZE} 行ごとに分割して書き込みます...")

    ws.update(range_name='A1', values=[values[0]])

    for i in range(1, total_rows, CHUNK_SIZE):
        chunk = values[i:i + CHUNK_SIZE]
        start_row = i + 1
        end_row = start_row + len(chunk) - 1

        range_name = f"A{start_row}"

        for retry in range(3):
            try:
                ws.update(range_name=range_name, values=chunk)
                print(f"書き込み中... 行 {start_row} 〜 {end_row}")
                time.sleep(1)
                break
            except APIError as e:
                print(f"APIエラー発生 (リトライ {retry+1}/3): {e}")
                time.sleep(5)
        else:
            print(f"行 {start_row} の書き込みに失敗しました。")

    print(f"統合完了！ すべてのデータを '{TARGET_SHEET_NAME}' に書き出しました。")
else:
    print("データが見つかりませんでした。")

合計 401 個のファイルを処理中（各銘柄 20 行抽出）...
処理完了: 成功 401 ファイル, スキップ/エラー 0 ファイル
全 8021 行のデータを 1000 行ごとに分割して書き込みます...
書き込み中... 行 2 〜 1001
書き込み中... 行 1002 〜 2001
書き込み中... 行 2002 〜 3001
書き込み中... 行 3002 〜 4001
書き込み中... 行 4002 〜 5001
書き込み中... 行 5002 〜 6001
書き込み中... 行 6002 〜 7001
書き込み中... 行 7002 〜 8001
書き込み中... 行 8002 〜 8021
統合完了！ すべてのデータを 'All_Stocks_Summary' に書き出しました。


In [3]:
import requests

# ステップ2でコピーしたURLを貼り付ける
url = "https://script.google.com/macros/s/AKfycbzvqo3jjM53w3UGGezgzFpXljFK4Rk77uePVsUhzde0Lw5McmMPyCBO-R4BnCmCUST3Gg/exec"

# リクエストを送る
try:
    response = requests.post(url)
    print("レスポンス:", response.text)
except Exception as e:
    print("エラーが発生しました:", e)

レスポンス: 処理が完了しました
